# Тестирование задач 2 и 3

Этот ноутбук тестирует реализацию расчета векторов доходностей и ковариационных матриц на готовых данных.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from task_2_3 import (
    load_prices_data,
    calculate_returns,
    calculate_exponential_weights,
    calculate_weighted_returns,
    calculate_weighted_covariance,
    rolling_window_analysis,
    expanding_window_analysis
)

# Настройка графиков
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 100)

## 1. Загрузка данных

In [ ]:
# Загрузка данных
prices = load_prices_data('data/prices_moex_new.csv')

print("=== Информация о данных ===")
print(f"Размер: {prices.shape}")
print(f"Период: {prices.index.min()} - {prices.index.max()}")
print(f"Количество акций: {len(prices.columns)}")
print(f"\nСписок акций: {list(prices.columns)}")
print("\nПервые 5 строк:")
print(prices.head())

## 2. Расчет доходностей

In [ ]:
# Расчет логарифмических доходностей
returns = calculate_returns(prices)

print("=== Информация о доходностях ===")
print(f"Размер: {returns.shape}")
print(f"Период: {returns.index.min()} - {returns.index.max()}")
print("\nОписательная статистика доходностей:")
print(returns.describe())

# Проверка на пропущенные значения
missing_values = returns.isnull().sum()
print(f"\nКоличество пропущенных значений по акциям:")
print(missing_values[missing_values > 0])

## 3. Визуализация доходностей

In [ ]:
# График кумулятивных доходностей для первых 5 акций
plt.figure(figsize=(12, 6))
for col in returns.columns[:5]:
    cumulative_returns = (1 + returns[col]).cumprod()
    plt.plot(returns.index, cumulative_returns, label=col)

plt.title('Кумулятивные доходности (первые 5 акций)')
plt.xlabel('Дата')
plt.ylabel('Кумулятивная доходность')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Тестирование экспоненциальных весов

In [ ]:
# Тест расчета экспоненциальных весов
n = 250  # Примерно 1 год торговых дней
lambda_param = 0.94

weights = calculate_exponential_weights(n, lambda_param)

print("=== Экспоненциальные веса ===")
print(f"Количество наблюдений: {n}")
print(f"Параметр lambda: {lambda_param}")
print(f"\nПервые 5 весов: {weights[:5]}")
print(f"Последние 5 весов: {weights[-5:]}")
print(f"Сумма весов: {weights.sum():.10f}")

# График весов
plt.figure(figsize=(10, 5))
plt.plot(weights)
plt.title('Экспоненциальные веса (lambda=0.94)')
plt.xlabel('Индекс наблюдения')
plt.ylabel('Вес')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Задача 2a: Скользящее окно

In [ ]:
# Анализ скользящим окном (1 год, шаг 1 год)
print("=== Задача 2a: Скользящее окно ===")
print("Окно: 1 год, Шаг: 1 год")

rolling_results = rolling_window_analysis(returns, window_size='1Y', step_size='1Y')

print(f"\nПолучено окон: {len(rolling_results)}")

# Анализ первого окна
if rolling_results:
    first_date = list(rolling_results.keys())[0]
    first_result = rolling_results[first_date]
    
    print(f"\n--- Первое окно ---")
    print(f"Период: {first_result['window_start'].strftime('%Y-%m-%d')} - {first_date.strftime('%Y-%m-%d')}")
    print(f"Размер окна: {len(first_result['window_returns'])} наблюдений")
    print(f"Размер ковариационной матрицы: {first_result['covariance_matrix'].shape}")
    
    # Проверка ковариационной матрицы
    cov_matrix = first_result['covariance_matrix']
    print(f"\nСвойства ковариационной матрицы:")
    print(f"- Симметричность: {np.allclose(cov_matrix, cov_matrix.T)}")
    print(f"- Определитель: {np.linalg.det(cov_matrix):.6e}")
    
    # Средние доходности для первых 5 акций
    mean_returns = first_result['mean_returns']
    print(f"\nСредние доходности (первые 5 акций):")
    for i, col in enumerate(returns.columns[:5]):
        print(f"  {col}: {mean_returns[i]:.6f}")

In [ ]:
# Динамика средних доходностей во времени
if rolling_results:
    dates = list(rolling_results.keys())
    mean_returns_history = [rolling_results[date]['mean_returns'] for date in dates]
    mean_returns_df = pd.DataFrame(mean_returns_history, index=dates, columns=returns.columns)
    
    # График динамики для первых 5 акций
    plt.figure(figsize=(12, 6))
    for col in returns.columns[:5]:
        plt.plot(mean_returns_df.index, mean_returns_df[col], marker='o', label=col)
    
    plt.title('Динамика средних доходностей (скользящее окно 1 год)')
    plt.xlabel('Дата')
    plt.ylabel('Средняя доходность')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Задача 2b: Расширяющееся окно

In [ ]:
# Анализ расширяющимся окном (шаг 1 год)
print("=== Задача 2b: Расширяющееся окно ===")
print("Шаг: 1 год")

expanding_results = expanding_window_analysis(returns, step_size='1Y')

print(f"\nПолучено окон: {len(expanding_results)}")

# Анализ динамики размера окна
if expanding_results:
    print("\n--- Динамика размера окна ---")
    for i, (date, result) in enumerate(list(expanding_results.items())[:5]):
        print(f"Окно {i+1}: {date.strftime('%Y-%m-%d')} - {result['window_size']} наблюдений")
    print("...")
    
    # Анализ последнего окна
    last_date = list(expanding_results.keys())[-1]
    last_result = expanding_results[last_date]
    print(f"\n--- Последнее окно ---")
    print(f"Период: {last_result['window_start'].strftime('%Y-%m-%d')} - {last_result['window_end'].strftime('%Y-%m-%d')}")
    print(f"Размер окна: {last_result['window_size']} наблюдений")

In [ ]:
# Сравнение скользящего и расширяющегося окон
if rolling_results and expanding_results:
    rolling_dates = list(rolling_results.keys())
    expanding_dates = list(expanding_results.keys())
    
    # Возьмем одну акцию для сравнения
    ticker = returns.columns[0]
    
    rolling_means = [rolling_results[date]['mean_returns'][0] for date in rolling_dates]
    expanding_means = [expanding_results[date]['mean_returns'][0] for date in expanding_dates]
    
    plt.figure(figsize=(12, 6))
    plt.plot(rolling_dates, rolling_means, marker='o', label='Скользящее окно')
    plt.plot(expanding_dates, expanding_means, marker='s', label='Расширяющееся окно')
    plt.title(f'Сравнение средних доходностей ({ticker})')
    plt.xlabel('Дата')
    plt.ylabel('Средняя доходность')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. Задача 3: Экспоненциальное забывание

In [ ]:
# Анализ скользящим окном с экспоненциальным забыванием
print("=== Задача 3: Экспоненциальное забывание ===")
print("Скользящее окно (1 год), lambda=0.94")

rolling_exp_results = rolling_window_analysis(
    returns, window_size='1Y', step_size='1Y', lambda_param=0.94
)

print(f"\nПолучено окон: {len(rolling_exp_results)}")

# Сравнение обычного и экспоненциального взвешивания
if rolling_results and rolling_exp_results:
    # Возьмем одну акцию для сравнения
    ticker = returns.columns[0]
    ticker_idx = 0
    
    dates = list(rolling_results.keys())
    rolling_means = [rolling_results[date]['mean_returns'][ticker_idx] for date in dates]
    exp_means = [rolling_exp_results[date]['mean_returns'][ticker_idx] for date in dates]
    
    plt.figure(figsize=(12, 6))
    plt.plot(dates, rolling_means, marker='o', label='Равномерное взвешивание')
    plt.plot(dates, exp_means, marker='s', label='Экспоненциальное забывание (λ=0.94)')
    plt.title(f'Сравнение методов взвешивания ({ticker})')
    plt.xlabel('Дата')
    plt.ylabel('Средняя доходность')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Численное сравнение
    print("\n--- Численное сравнение для первой даты ---")
    first_date = dates[0]
    print(f"Дата: {first_date.strftime('%Y-%m-%d')}")
    print(f"Равномерное взвешивание: {rolling_means[0]:.6f}")
    print(f"Экспоненциальное забывание: {exp_means[0]:.6f}")
    print(f"Разница: {exp_means[0] - rolling_means[0]:.6f}")

## 8. Визуализация ковариационной матрицы

In [ ]:
# Тепловая карта корреляционной матрицы для первого окна
if rolling_results:
    first_date = list(rolling_results.keys())[0]
    cov_matrix = rolling_results[first_date]['covariance_matrix']
    
    # Расчет корреляционной матрицы
    stds = np.sqrt(np.diag(cov_matrix))
    corr_matrix = cov_matrix / np.outer(stds, stds)
    
    # Тепловая карта для первых 10 акций
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        corr_matrix[:10, :10],
        annot=True,
        fmt='.2f',
        cmap='coolwarm',
        center=0,
        xticklabels=returns.columns[:10],
        yticklabels=returns.columns[:10]
    )
    plt.title(f'Корреляционная матрица (окно от {first_date.strftime("%Y-%m-%d")})')
    plt.tight_layout()
    plt.show()

## 9. Дополнительные размеры окон

In [ ]:
# Тестирование различных размеров окон
window_sizes = ['1M', '1Q', '1Y']

print("=== Тестирование различных размеров окон ===")
for window_size in window_sizes:
    results = rolling_window_analysis(returns, window_size=window_size, step_size=window_size)
    print(f"\nОкно {window_size}: {len(results)} окон")

## 10. Итоговый анализ

In [ ]:
# Сводная статистика по всем методам
print("=== Итоговый анализ ===")
print(f"\nПериод данных: {returns.index.min()} - {returns.index.max()}")
print(f"Всего наблюдений: {len(returns)}")
print(f"Количество акций: {len(returns.columns)}")

if rolling_results:
    print(f"\nСкользящее окно (1 год): {len(rolling_results)} окон")
    
if expanding_results:
    print(f"Расширяющееся окно: {len(expanding_results)} окон")
    
if rolling_exp_results:
    print(f"Скользящее окно с эксп. забыванием: {len(rolling_exp_results)} окон")

print("\n=== Проверка корректности ===")
# Проверка, что ковариационные матрицы симметричны
all_symmetric = True
for date, result in rolling_results.items():
    cov = result['covariance_matrix']
    if not np.allclose(cov, cov.T):
        all_symmetric = False
        print(f"Ошибка: матрица для {date} не симметрична")
        break

if all_symmetric:
    print("✓ Все ковариационные матрицы симметричны")

# Проверка положительной определенности (для первых 5 окон)
pd_positive = True
for i, (date, result) in enumerate(list(rolling_results.items())[:5]):
    cov = result['covariance_matrix']
    try:
        np.linalg.cholesky(cov)
    except np.linalg.LinAlgError:
        pd_positive = False
        print(f"Предупреждение: матрица для {date} не положительно определена")
        break

if pd_positive:
    print("✓ Первые 5 ковариационных матриц положительно определены")

## Вывод

В этом ноутбуке мы протестировали:
1. Загрузку и подготовку данных о ценах
2. Расчет логарифмических доходностей
3. Реализацию скользящего окна (Задача 2a)
4. Реализацию расширяющегося окна (Задача 2b)
5. Реализацию экспоненциального забывания (Задача 3)
6. Корректность ковариационных матриц

Все функции работают корректно и могут быть использованы для дальнейшего анализа.